# Chapter 5: Bond Curve Analysis Module

This chapter covers bond ETF analysis:
- Bond curve data loading
- Duration and credit analysis
- ETF performance simulation
- Yield curve visualization

## 1. Setup and Imports

In [ ]:
import datetime
import os
import sys
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.dirname(os.path.abspath('__file__'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import BOND_DB, PERFORMANCE_CONFIG, setup_logging
from utils import DataLoader, PerformanceAnalyzer

logger = setup_logging('bond_curve_analysis')

## 2. Bond Types and Categories

In [ ]:
# Bond type definitions
BOND_TYPES = {
    'CD': 'Certificate of Deposit',
    'Financial': 'Financial Bond',
    'Tier2': 'Tier 2 Capital Bond',
    'Municipal': 'Municipal/Chengtou Bond',
    'Corporate': 'Corporate Bond',
    'ABS': 'Asset-Backed Security',
    'Treasury': 'Treasury Bond'
}

# Duration categories (in years)
DURATION_TERMS = [0.5, 1, 2, 3, 4, 5, 7, 10, 15, 20, 30]

# Credit rating categories
CREDIT_RATINGS = ['High Quality', 'Low Quality', 'All']

print("Bond Categories:")
for code, name in BOND_TYPES.items():
    print(f"  {code}: {name}")

## 3. Bond Curve Analyzer Class

In [ ]:
class BondCurveAnalyzer:
    """Bond curve analysis class"""
    
    def __init__(self, data_loader: DataLoader = None):
        """Initialize bond curve analyzer
        
        Args:
            data_loader: DataLoader instance
        """
        self.data_loader = data_loader or DataLoader()
        self.analyzer = PerformanceAnalyzer()
    
    def get_bond_curve_data(self, start_date: str, end_date: str = None,
                            bond_type: str = None, term: float = None) -> pd.DataFrame:
        """Get bond curve data
        
        Args:
            start_date: Start date
            end_date: End date
            bond_type: Bond type filter
            term: Duration term filter
            
        Returns:
            DataFrame with curve data
        """
        return self.data_loader.get_bond_curve_data(
            start_date, end_date, bond_type, term
        )
    
    def simulate_bond_etf(self, bond_type: str, term: float = None,
                          credit_rating: str = None,
                          start_date: str = None, 
                          end_date: str = None) -> dict:
        """Simulate bond ETF performance
        
        Args:
            bond_type: Bond type
            term: Duration term
            credit_rating: Credit rating
            start_date: Start date
            end_date: End date
            
        Returns:
            Simulation results dictionary
        """
        # Get curve data
        curve_data = self.get_bond_curve_data(
            start_date, end_date, bond_type, term
        )
        
        if curve_data.empty:
            return None
        
        # Sort by date
        curve_data = curve_data.sort_values('dt')
        
        # Calculate daily returns (yield decrease = price increase)
        curve_data['daily_return'] = -curve_data['yield'].diff() / 100
        
        # Calculate NAV
        curve_data['nav'] = (1 + curve_data['daily_return']).cumprod()
        
        # Calculate metrics
        returns = curve_data['daily_return'].dropna()
        nav = curve_data['nav'].dropna()
        
        metrics = {
            'returns': self.analyzer.calculate_returns(nav),
            'volatility': self.analyzer.calculate_volatility(returns),
            'max_drawdown': self.analyzer.calculate_max_drawdown(nav),
            'sharpe_ratio': self.analyzer.calculate_sharpe_ratio(returns)
        }
        
        return {
            'nav': nav,
            'returns': returns,
            'metrics': metrics,
            'bond_type': bond_type,
            'term': term,
            'credit_rating': credit_rating
        }
    
    def generate_bond_etf_list(self) -> pd.DataFrame:
        """Generate list of simulated bond ETFs
        
        Returns:
            DataFrame with bond ETF list
        """
        etf_list = []
        
        for bond_type in BOND_TYPES.keys():
            for term in DURATION_TERMS:
                for rating in CREDIT_RATINGS:
                    etf_code = f'BOND_{bond_type}_{term}Y_{rating}'.replace('.', 'P')
                    etf_list.append({
                        'etf_code': etf_code,
                        'bond_type': bond_type,
                        'bond_type_name': BOND_TYPES[bond_type],
                        'term': term,
                        'credit_rating': rating
                    })
        
        return pd.DataFrame(etf_list)
    
    def analyze_duration_performance(self, bond_type: str,
                                     start_date: str) -> pd.DataFrame:
        """Analyze performance across different durations
        
        Args:
            bond_type: Bond type to analyze
            start_date: Start date for analysis
            
        Returns:
            DataFrame with duration analysis
        """
        results = []
        
        for term in DURATION_TERMS:
            sim_result = self.simulate_bond_etf(
                bond_type=bond_type,
                term=term,
                start_date=start_date
            )
            
            if sim_result:
                results.append({
                    'term': term,
                    'annual_return': sim_result['metrics']['returns'] * 100,
                    'volatility': sim_result['metrics']['volatility'] * 100,
                    'max_drawdown': sim_result['metrics']['max_drawdown'] * 100,
                    'sharpe_ratio': sim_result['metrics']['sharpe_ratio']
                })
        
        return pd.DataFrame(results)

# Initialize bond curve analyzer
bond_analyzer = BondCurveAnalyzer()
print("Bond curve analyzer initialized")

# Generate bond ETF list
bond_etf_list = bond_analyzer.generate_bond_etf_list()
print(f"\nGenerated {len(bond_etf_list)} bond ETF definitions")
bond_etf_list.head(10)

## 4. Yield Curve Analysis

In [ ]:
def analyze_yield_curve(curve_data: pd.DataFrame) -> dict:
    """Analyze yield curve characteristics
    
    Args:
        curve_data: DataFrame with curve data
        
    Returns:
        Analysis results dictionary
    """
    if curve_data.empty:
        return {}
    
    analysis = {
        'date_range': {
            'start': curve_data['dt'].min(),
            'end': curve_data['dt'].max()
        },
        'yield_stats': {
            'mean': curve_data['yield'].mean(),
            'std': curve_data['yield'].std(),
            'min': curve_data['yield'].min(),
            'max': curve_data['yield'].max()
        },
        'term_structure': {}
    }
    
    # Term structure analysis
    if 'term' in curve_data.columns:
        term_yields = curve_data.groupby('term')['yield'].mean()
        analysis['term_structure'] = term_yields.to_dict()
    
    return analysis

def calculate_curve_steepness(curve_data: pd.DataFrame, 
                              short_term: float = 2, 
                              long_term: float = 10) -> pd.Series:
    """Calculate yield curve steepness
    
    Args:
        curve_data: DataFrame with curve data
        short_term: Short-term duration
        long_term: Long-term duration
        
    Returns:
        Series with steepness values
    """
    short_yields = curve_data[curve_data['term'] == short_term].set_index('dt')['yield']
    long_yields = curve_data[curve_data['term'] == long_term].set_index('dt')['yield']
    
    steepness = long_yields - short_yields
    return steepness

print("Yield curve analysis functions defined")

## 5. Duration Strategy Analysis

In [ ]:
def compare_duration_strategies(bond_type: str, start_date: str,
                                end_date: str = None) -> pd.DataFrame:
    """Compare different duration strategies
    
    Args:
        bond_type: Bond type
        start_date: Start date
        end_date: End date
        
    Returns:
        DataFrame with strategy comparison
    """
    analyzer = BondCurveAnalyzer()
    
    # Define duration categories
    duration_categories = {
        'Ultra Short': [0.5, 1],
        'Short': [1, 2],
        'Medium': [3, 5],
        'Long': [7, 10],
        'Ultra Long': [15, 20, 30]
    }
    
    results = []
    
    for category, terms in duration_categories.items():
        for term in terms:
            sim = analyzer.simulate_bond_etf(
                bond_type=bond_type,
                term=term,
                start_date=start_date,
                end_date=end_date
            )
            
            if sim:
                results.append({
                    'category': category,
                    'term': term,
                    'annual_return': sim['metrics']['returns'] * 100,
                    'volatility': sim['metrics']['volatility'] * 100,
                    'sharpe': sim['metrics']['sharpe_ratio']
                })
    
    return pd.DataFrame(results)

def get_optimal_duration(curve_data: pd.DataFrame, 
                        risk_tolerance: str = 'medium') -> float:
    """Determine optimal duration based on risk tolerance
    
    Args:
        curve_data: DataFrame with curve data
        risk_tolerance: Risk tolerance level
        
    Returns:
        Optimal duration
    """
    # Risk tolerance to duration mapping
    duration_map = {
        'low': 1.0,      # Short duration
        'medium': 3.0,   # Medium duration
        'high': 7.0      # Long duration
    }
    
    return duration_map.get(risk_tolerance, 3.0)

print("Duration strategy functions defined")

## 6. Credit Analysis

In [ ]:
def analyze_credit_spread(curve_data: pd.DataFrame) -> pd.DataFrame:
    """Analyze credit spreads
    
    Args:
        curve_data: DataFrame with curve data
        
    Returns:
        DataFrame with credit spread analysis
    """
    if 'credit_rating' not in curve_data.columns:
        return pd.DataFrame()
    
    # Group by credit rating and term
    spread_analysis = curve_data.groupby(['credit_rating', 'term']).agg({
        'yield': ['mean', 'std', 'min', 'max']
    }).reset_index()
    
    spread_analysis.columns = ['credit_rating', 'term', 'mean_yield', 
                               'yield_std', 'min_yield', 'max_yield']
    
    return spread_analysis

def compare_credit_strategies(bond_type: str, term: float,
                              start_date: str) -> pd.DataFrame:
    """Compare different credit strategies
    
    Args:
        bond_type: Bond type
        term: Duration term
        start_date: Start date
        
    Returns:
        DataFrame with credit strategy comparison
    """
    analyzer = BondCurveAnalyzer()
    results = []
    
    for rating in ['High Quality', 'Low Quality', 'All']:
        sim = analyzer.simulate_bond_etf(
            bond_type=bond_type,
            term=term,
            credit_rating=rating,
            start_date=start_date
        )
        
        if sim:
            results.append({
                'credit_rating': rating,
                'annual_return': sim['metrics']['returns'] * 100,
                'volatility': sim['metrics']['volatility'] * 100,
                'sharpe': sim['metrics']['sharpe_ratio']
            })
    
    return pd.DataFrame(results)

print("Credit analysis functions defined")

## 7. Visualization

In [ ]:
def plot_yield_curve(curve_data: pd.DataFrame, date: str = None):
    """Plot yield curve
    
    Args:
        curve_data: DataFrame with curve data
        date: Specific date to plot (optional)
        
    Returns:
        Plotly Figure object
    """
    fig = go.Figure()
    
    if date:
        plot_data = curve_data[curve_data['dt'] == date]
    else:
        # Use latest date
        latest_date = curve_data['dt'].max()
        plot_data = curve_data[curve_data['dt'] == latest_date]
    
    fig.add_trace(go.Scatter(
        x=plot_data['term'],
        y=plot_data['yield'],
        mode='lines+markers',
        name='Yield Curve'
    ))
    
    fig.update_layout(
        title='Yield Curve',
        xaxis_title='Duration (Years)',
        yaxis_title='Yield (%)',
        template='plotly_white'
    )
    
    return fig

def plot_duration_performance(comparison_df: pd.DataFrame):
    """Plot duration strategy performance
    
    Args:
        comparison_df: DataFrame with duration comparison
        
    Returns:
        Plotly Figure object
    """
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Annual Return', 'Volatility', 
                       'Sharpe Ratio', 'Return vs Risk'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}],
               [{'type': 'bar'}, {'type': 'scatter'}]]
    )
    
    # Annual return
    fig.add_trace(
        go.Bar(x=comparison_df['term'], y=comparison_df['annual_return'],
               name='Return'),
        row=1, col=1
    )
    
    # Volatility
    fig.add_trace(
        go.Bar(x=comparison_df['term'], y=comparison_df['volatility'],
               name='Volatility'),
        row=1, col=2
    )
    
    # Sharpe ratio
    fig.add_trace(
        go.Bar(x=comparison_df['term'], y=comparison_df['sharpe'],
               name='Sharpe'),
        row=2, col=1
    )
    
    # Return vs Risk scatter
    fig.add_trace(
        go.Scatter(x=comparison_df['volatility'], 
                   y=comparison_df['annual_return'],
                   mode='markers+text',
                   text=comparison_df['term'],
                   name='Risk-Return'),
        row=2, col=2
    )
    
    fig.update_layout(
        title='Duration Strategy Performance Comparison',
        height=800,
        showlegend=False,
        template='plotly_white'
    )
    
    return fig

print("Visualization functions defined")

## 8. Municipal Bond Analysis

In [ ]:
def analyze_municipal_bonds(curve_data: pd.DataFrame, 
                            province: str = None) -> pd.DataFrame:
    """Analyze municipal (Chengtou) bond performance
    
    Args:
        curve_data: DataFrame with curve data
        province: Province filter (optional)
        
    Returns:
        DataFrame with municipal bond analysis
    """
    if curve_data.empty:
        return pd.DataFrame()
    
    # Filter for municipal bonds
    muni_data = curve_data[curve_data['bond_type'] == 'Municipal']
    
    if province and 'province' in muni_data.columns:
        muni_data = muni_data[muni_data['province'] == province]
    
    # Group by term
    results = []
    for term in [1, 3, 5, 7, 10]:
        term_data = muni_data[muni_data['term'] == term]
        
        if not term_data.empty:
            results.append({
                'term': term,
                'avg_yield': term_data['yield'].mean(),
                'yield_volatility': term_data['yield'].std() * np.sqrt(252),
                'sample_size': len(term_data)
            })
    
    return pd.DataFrame(results)

print("Municipal bond analysis function defined")

## 9. Summary

This chapter covered:
1. Bond curve data loading and analysis
2. Bond ETF performance simulation
3. Duration strategy comparison
4. Credit spread analysis
5. Yield curve visualization
6. Municipal bond analysis

### Key Classes
- `BondCurveAnalyzer`: Main bond analysis class

### Key Functions
- `analyze_yield_curve`: Yield curve characteristics
- `compare_duration_strategies`: Duration comparison
- `analyze_credit_spread`: Credit spread analysis
- `plot_yield_curve`: Visualization

### Best Practices
- Consider duration risk in bond investing
- Monitor credit spreads for opportunities
- Use curve steepness as economic indicator
- Diversify across duration and credit